# DriveInsight

## About

DriveInsight is a large-scale, CCTV-based traffic dataset that bridges the gap between real-world traffic observations and simulation-ready scenarios. Scenarios are extracted from 24/7 CCTV camera feeds at urban intersections across more than 20 locations in multiple countries, then converted into standardized OpenSCENARIO (.xosc) format. Each scenario ships with a corresponding OpenDRIVE (.xodr) road network, weather metadata, and temporal annotations, making it directly compatible with simulation tools such as esmini, CARLA, and VTD.

The dataset covers a diverse range of traffic participants and is designed for trajectory forecasting, scenario-based ADAS validation, multi-agent interaction modelling, and long-tail behaviour analysis.

### How to Obtain the Dataset

The DriveInsight dataset is publicly available on GitHub. Clone the repository directly to get started:

```bash
git clone https://github.com/Drive1nsight/driveinsightD.git
cd driveinsightD
```

For extended intersections, additional scenario collections, and high-resolution CCTV source footage, submit a request via the project website at [https://driveinsight.io](https://driveinsight.io).

### How to Obtain the Map

Road network files in OpenDRIVE (.xodr) format are bundled with the dataset under each location folder:

```
driveinsightD/
└── database/
    └── <location_id>/
        ├── {id}_scenario.xosc          # OpenSCENARIO trajectory file
        ├── {id}_scenario_metadata.json # Scenario metadata
        ├── <location_id>.xodr          # OpenDRIVE road network
        └── <location_id>.osgb          # 3D environment model
```

No separate download is required — the map file for each location is located in the same folder as the scenario files.

### Citation

```latex
@article{zhdanov2025driveinsight,
  title={DriveInsight: CCTV-based dataset capturing real-world scenarios},
  author={Zhdanov, Pavlo and Yerumbayev, Sultan and Khan, Adil and Ram{\'i}rez Rivera, Ad{\'i}n},
  journal={IEEE Transactions on Circuits and Systems for Video Technology},
  year={2025},
  note={Under review}
}
```

## Data Analysis

> This part is independently conducted by Tactics2D.

### Trajectory Types in DriveInsight

- `Car`
- `Truck`
- `Bus`
- `Motorcycle`
- `Bicycle`
- `Pedestrian`

### Distribution of Trajectory Categories

DriveInsight covers six types of traffic participants: Car, Truck, Bus, Motorcycle, Bicycle, and Pedestrian. 
The composition varies across locations, reflecting the diversity of urban intersection traffic across different countries.

In [18]:
import os
import matplotlib.pyplot as plt
from collections import Counter
from tactics2d.dataset_parser import DriveInsightDParser

_parser = DriveInsightDParser()

location_configs = [
    ("Zlin, Czech Republic",   os.environ.get("DRIVEINSIGHTD_CZ_FOLDER", "../../tactics2d/data/trajectory_sample/DriveInsightD/cz_zlin"),    "11",   "cz_zlin.xodr"),
    ("Taito, Japan",           os.environ.get("DRIVEINSIGHTD_JP_FOLDER", "../../tactics2d/data/trajectory_sample/DriveInsightD/jp_taito"),    "6",    "jp_taito.xodr"),
    ("Coldwater, United States", os.environ.get("DRIVEINSIGHTD_US_FOLDER", "../../tactics2d/data/trajectory_sample/DriveInsightD/us_coldwater"), "4464", "usa_coldwater.xodr"),
]

type_order = ["car", "truck", "bus", "motorcycle", "bicycle", "pedestrian", "other"]
colors     = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860", "#AAAAAA"]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("Distribution of Trajectory Categories across DriveInsightD Locations", fontsize=8)

for ax, (loc_name, folder, scenario_id, map_name) in zip(axes, location_configs):
    try:
        participants, _ = _parser.parse_trajectory(file=scenario_id, folder=folder)
        type_counts = Counter()
        for p in participants.values():
            cls = type(p).__name__.lower()
            t = {"vehicle": "car", "cyclist": "bicycle", "pedestrian": "pedestrian", "other": "other"}.get(cls, "other")
            type_counts[t.lower()] += 1

        labels  = [t for t in type_order if type_counts[t] > 0]
        values  = [type_counts[t] for t in labels]
        clrs    = [colors[type_order.index(t)] for t in labels]

        ax.pie(values, labels=labels, colors=clrs, autopct='%1.1f%%', startangle=90,
               textprops={'fontsize': 6})
        ax.set_title(f"{loc_name}\n(scenario {scenario_id}, n={sum(values)})", fontsize=7)
    except Exception as e:
        ax.text(0.5, 0.5, f"Data not available\n({e})", ha='center', va='center',
                transform=ax.transAxes, fontsize=6)
        ax.set_title(loc_name, fontsize=7)

plt.tight_layout()
plt.show()


In [19]:
import io
import os
import sys
import warnings

# ---------------------------------------------------------------------------
# Dataset paths — update these to match your local installation
# ---------------------------------------------------------------------------
os.environ["DRIVEINSIGHTD_FOLDER"]    = "/root/autodl-tmp/driveinsightD/database/cz_zlin"
os.environ["DRIVEINSIGHTD_CZ_FOLDER"] = "/root/autodl-tmp/driveinsightD/database/cz_zlin"
os.environ["DRIVEINSIGHTD_JP_FOLDER"] = "/root/autodl-tmp/driveinsightD/database/jp_taito"
os.environ["DRIVEINSIGHTD_US_FOLDER"] = "/root/autodl-tmp/driveinsightD/database/us_coldwater"

warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR,force=True)


# Required for off-screen rendering in notebook environments
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

sys.path.insert(0, "/root/autodl-tmp/tactics2d")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from IPython.display import Image as IPImage, display

In [20]:
mpl.rcParams.update(
    {
        "figure.dpi": 150,
        "font.family": "DejaVu Sans Mono",
        "font.size": 6,
        "font.stretch": "semi-expanded",
        "animation.html": "jshtml",
        "animation.embed_limit": 5000,
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8,
        "axes.facecolor": "white",
    }
)


def render_scenario_animation(scenario: dict, window_size=(640, 640), step_ms: int = 250):
    """Render a scenario as an HTML5 animation.

    Args:
        scenario (dict): Output of DriveInsightDParser.parse().
        window_size (tuple): Width and height of the rendered frame in pixels.
        step_ms (int): Time step between frames in milliseconds.

    Returns:
        IPython.display.HTML: Inline-displayable HTML5 animation.
    """
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML
    from tactics2d.sensor import BEVCamera
    from tactics2d.participant.element import Vehicle, Cyclist, Pedestrian, Other
    from tactics2d.participant.trajectory import Trajectory, State
    from shapely.geometry import Point

    map_         = scenario["map"]
    participants = scenario["participants"]
    t_start, t_end = scenario["time_range"]

    # Re-key participants with integer IDs (BEVCamera requires int IDs)
    int_participants = {}
    for idx, (name, p) in enumerate(participants.items()):
        class_ = type(p)
        new_p = class_(
            id_=idx,
            type_=p.type_,
            length=p.length,
            width=p.width,
            trajectory=p.trajectory,
        )
        new_p.trajectory.id_ = idx
        int_participants[idx] = new_p

    all_ts = set()
    for p in int_participants.values():
        frames = getattr(p.trajectory, "frames", None)
        if frames is not None:
            all_ts.update(frames)
    times = sorted(t for t in all_ts if t_start <= t <= t_end)[::max(1, step_ms // 50)]

    boundary = map_.boundary
    x_min, x_max, y_min, y_max = boundary
    camera_position = np.array([(x_min + x_max) / 2, (y_min + y_max) / 2])

    camera = BEVCamera(id_=0, map_=map_)
    prev_road_id_set = set()
    prev_participant_id_set = set()

    from tactics2d.renderer import MatplotlibRenderer
    renderer = MatplotlibRenderer(
        xlim=(x_min, x_max), ylim=(y_min, y_max),
        resolution=window_size, auto_scale=True
    )
    fig = renderer.fig

    def update(t):
        nonlocal prev_road_id_set, prev_participant_id_set
        participant_ids = [pid for pid, p in int_participants.items() if p.is_active(t)]
        geometry_data, prev_road_id_set, prev_participant_id_set = camera.update(
            t, int_participants, participant_ids,
            prev_road_id_set, prev_participant_id_set,
            Point(camera_position),
        )
        renderer.update(geometry_data)
        renderer.ax.set_title(f"t = {t} ms  |  active: {len(participant_ids)}", fontsize=6)

    ani = FuncAnimation(fig, update, frames=times, interval=100, repeat=True)
    plt.close(fig)
    return HTML(ani.to_html5_video())


### Parse Trajectory

In [21]:
from tactics2d.dataset_parser import DriveInsightDParser

dataset_path = os.environ.get(
    "DRIVEINSIGHTD_FOLDER",
    "../../tactics2d/data/trajectory_sample/DriveInsightD/cz_zlin",
)
map_name = os.environ.get("DRIVEINSIGHTD_MAP", "cz_zlin.xodr")

dataset_parser = DriveInsightDParser()

participants, time_range = dataset_parser.parse_trajectory(
    file="106", folder=dataset_path
)

print(f"Number of participants : {len(participants)}")
print(f"Time range             : {time_range[0]} ms -> {time_range[1]} ms")

Number of participants : 11
Time range             : 0 ms -> 14750 ms


### Parse Map

In [22]:
from tactics2d.map.parser import XODRParser

map_parser = XODRParser()
map_ = map_parser.parse(file_path=os.path.join(dataset_path, map_name))

print(f"Number of lanes     : {len(map_.lanes)}")
print(f"Number of roadlines : {len(map_.roadlines)}")
print(f"Number of junctions : {len(map_.junctions)}")

Number of lanes     : 69
Number of roadlines : 92
Number of junctions : 1


### Load Full Scenario

In [23]:
scenario = dataset_parser.parse(
    scenario_id="106",
    folder=dataset_path,
    map_name=map_name,
)

print(f"Scenario ID   : {scenario['scenario_id']}")
print(f"Weather       : {scenario['metadata']['weather']}")
print(f"Precipitation : {scenario['metadata']['precipitation']}")
print(f"Recorded at   : {scenario['metadata']['time']}")
print(f"Road friction : {scenario['metadata']['friction']}")
print(f"Participants  : {len(scenario['participants'])}")
print(f"Time range    : {scenario['time_range'][0]} ms -> {scenario['time_range'][1]} ms")

Scenario ID   : 106
Weather       : free
Precipitation : dry
Recorded at   : 2020-03-20T12:00:00
Road friction : 1.0
Participants  : 11
Time range    : 0 ms -> 14750 ms


### Visualisation

The following animations show top-down renderings of three DriveInsightD locations parsed and visualised using Tactics2D. 
Each location uses a distinct OpenDRIVE road network, demonstrating the parser's ability to handle diverse intersection geometries across different countries.

In [24]:
# Zlin, Czech Republic — scenario 11 (cz_zlin.xodr)
dataset_parser = DriveInsightDParser()
cz_folder = os.environ.get(
    "DRIVEINSIGHTD_CZ_FOLDER",
    "../../tactics2d/data/trajectory_sample/DriveInsightD/cz_zlin",
)
cz_scenario = dataset_parser.parse(
    scenario_id="11",
    folder=cz_folder,
    map_name="cz_zlin.xodr",
)
print("Zlin, Czech Republic — scenario 11")
print(f"  Participants : {len(cz_scenario['participants'])}")
print(f"  Lanes        : {len(cz_scenario['map'].lanes)}")
render_scenario_animation(cz_scenario)


Zlin, Czech Republic — scenario 11
  Participants : 21
  Lanes        : 69


In [25]:
# Taito, Japan — scenario 6 (jp_taito.xodr)
jp_folder = os.environ.get(
    "DRIVEINSIGHTD_JP_FOLDER",
    "../../tactics2d/data/trajectory_sample/DriveInsightD/jp_taito",
)
jp_scenario = dataset_parser.parse(
    scenario_id="6",
    folder=jp_folder,
    map_name="jp_taito.xodr",
)
print("Taito, Japan — scenario 6")
print(f"  Participants : {len(jp_scenario['participants'])}")
print(f"  Lanes        : {len(jp_scenario['map'].lanes)}")
render_scenario_animation(jp_scenario)


Taito, Japan — scenario 6
  Participants : 13
  Lanes        : 79


In [26]:
# Coldwater, United States — scenario 4464 (usa_coldwater.xodr)
us_folder = os.environ.get(
    "DRIVEINSIGHTD_US_FOLDER",
    "../../tactics2d/data/trajectory_sample/DriveInsightD/us_coldwater",
)
us_scenario = dataset_parser.parse(
    scenario_id="4464",
    folder=us_folder,
    map_name="usa_coldwater.xodr",
)
print("Coldwater, United States — scenario 4464")
print(f"  Participants : {len(us_scenario['participants'])}")
print(f"  Lanes        : {len(us_scenario['map'].lanes)}")
render_scenario_animation(us_scenario)


Coldwater, United States — scenario 4464
  Participants : 16
  Lanes        : 234


## Appendix: Data Format

<div class="admonition note">
    <p class="admonition-title">Note</p>
    <p>
        This is a summary of the DriveInsight dataset format based on
        <a href="https://github.com/Drive1nsight/driveinsightD" target="_blank" rel="noopener noreferrer">
            the official repository
        </a>, provided here for reference purposes only.
    </p>
</div>

### Statistics

#### Coverage

Locations: 20+ urban intersections across multiple countries and continents  
Source footage: Hundreds of hours of 24/7 CCTV streams  
Scenario format: OpenSCENARIO (.xosc)  
Map format: OpenDRIVE (.xodr)

#### Agent Types and Count

| Agent Type   | Approximate Count |
|--------------|------------------|
| Car          | ~120,000         |
| Truck        | ~17,500          |
| Bus          | ~2,700           |
| Pedestrian   | ~55,000          |
| Bicycle      | ~34,000          |
| Motorcycle   | ~650             |

### Description of the Data and File Structure

Each scenario is stored as a pair of files under `database/<location_id>/`:

- **`{id}_scenario.xosc`**: OpenSCENARIO XML file defining all agent entities, their dimensions, and trajectory waypoints in world coordinates.
- **`{id}_scenario_metadata.json`**: JSON file containing scenario-level annotations:

```json
{
  "camera_name": "cz_zlin",
  "datetime_utc": "2025-10-16T18:21:45+00:00",
  "location_name": "Zlin, Czech Republic",
  "road_description": {
    "road_name": "Tr. Tomase Bati / Osvoboditelu",
    "road_type": "Urban Road",
    "road_condition": "Good",
    "traffic_density": "Moderate",
    "configuration": "crossing"
  },
  "weather": {
    "lat": 49.2256,
    "lon": 17.6687,
    "timezone": "Europe/Prague"
  }
}
```

- **`<location_id>.xodr`**: OpenDRIVE road network defining lane geometry, intersections, and road topology for the recording location.
- **`<location_id>.osgb`**: OpenSceneGraph binary 3D environment model for visualisation in esmini and compatible simulators.

### Simulator Compatibility

DriveInsight scenarios are compatible with the following industry-standard simulation tools:

- [esmini](https://github.com/esmini/esmini)
- [CARLA](https://carla.org)
- [VTD](https://www.mscsoftware.com/product/virtual-test-drive)